In [ ]:
import requests
import time
import json
import csv
import re
import os
from pathlib import Path
import pandas as pd

In [ ]:
API_TOKEN = "<SEMGEP_API_TOKEN>"
HEADERS = {"Authorization": f"Bearer {API_TOKEN}"}
url = "https://semgrep.dev/api/v1/deployments"

OUTPUT_CSV_PATH = "all_semgrep_findings.csv"
PAGE_SIZE = 3000
SLEEP_SECONDS = 0.5

resp = requests.get(url, headers=HEADERS)

if resp.status_code == 200:
    print(resp.json())
else:
    print("error", resp.status_code)
    

In [ ]:
DEPLOYMENT_SLUG = "<DEPLOYMENT_SLUG>"  
BASE_URL = f"https://semgrep.dev/api/v1/deployments/{DEPLOYMENT_SLUG}/findings?"


In [ ]:
def normalize_cwe_name(cwe_names):
    if not cwe_names:
        #print("no existing CWE name")
        return None

    raw_cwe = cwe_names[0] 

    match = re.search(r"CWE-(\d+)", raw_cwe) 
    if match:
        cwe_id = f"cwe-{match.group(1)}".lower()
        return cwe_id

    print("not a valid CWE-ID")
    return None


In [ ]:
def extract_groundtruth_and_filename(file_path):
    try:
        path = Path(file_path)
        parts = path.parts
        cwe_part = next(p for p in parts if p.lower().startswith("cwe_"))
        groundtruth = cwe_part.lower().replace("_", "-")
        filename = path.name
        return groundtruth, filename
        
    except Exception as e:
        print(f"error for extract_groundtruth_and_filename for {file_path}: {e}")
        return None, None


In [ ]:
def transform_findings(findings):
    rows = []

    for f in findings:
        finding_id = f.get("id")
        file_path = f.get("location", {}).get("file_path")

        if not file_path:
            print(f"no path found for {finding_id}")
            continue

        groundtruth_cwe_id, filename = extract_groundtruth_and_filename(file_path)
        #print(f"found cwe name: {f.get("rule", {}).get("cwe_names")}")
        found_cwe_id = normalize_cwe_name(f.get("rule", {}).get("cwe_names"))
        applied_rule = f.get("rule_name")
        status = f.get("status")

        rows.append({
            "finding_id": finding_id,
            "file_path": file_path,
            "groundtruth_cwe_id": groundtruth_cwe_id,
            "filename": filename,
            "found_cwe_id": found_cwe_id,
            "applied_semgrep_rule": applied_rule,
            "status": status,
        })

    return rows
    

In [ ]:
def save_findings_to_csv(findings, path, write_header=False):
    if not findings:
        return
    
    keys = findings[0].keys()

    with open(path, mode="a", newline="", encoding="utf-8") as f:
        csv_writer = csv.DictWriter(f, fieldnames=keys)
        if write_header:
            csv_writer.writeheader()
        csv_writer.writerows(findings)
        f.flush() 


In [ ]:
def fetch_findings():
    findings = []
    page = 0
    first_write = not os.path.exists(OUTPUT_CSV_PATH) 

    while True:
        print(f"loading page: {page}")

        params = {
            "page": page,
            "page_size": PAGE_SIZE,
        }

        response = requests.get(BASE_URL, headers=HEADERS, params=params, timeout=30)

        if response.status_code != 200:
            print(f"error for {page}: {response.status_code}")
            break

        data = response.json()
        page_findings = data.get("findings", [])

        if not page_findings:
            print("done")
            break

        transformed = transform_findings(page_findings)

        save_findings_to_csv(
            findings=transformed,
            path=OUTPUT_CSV_PATH,
            write_header=first_write
        )

        first_write = False
        page += 1
        time.sleep(SLEEP_SECONDS)


In [ ]:
fetch_findings()

In [ ]:
df = pd.read_csv(OUTPUT_CSV_PATH)  
print(df.info())

unique_finding_count = df["finding_id"].nunique()
print(f"number of unique finding-ids: {unique_finding_count}")


In [ ]:
df_unique = df.drop_duplicates()
df_unique.to_csv("all_semgrep_findings_cleaned.csv", index=False) 

print(f"{len(df) - len(df_unique)} double rows deleted")
print(f"{len(df_unique)} unique rows saved")
